# Notebook 08 – Ensemble, chọn hệ thống cuối và quy trình nộp bài

**Mục tiêu**
1. Thử ghép nhiều model (ensemble) để xem có hơn model đơn lẻ tốt nhất không, và đo điều đó **trung thực**.
2. Chốt "công thức" hệ thống cuối cho A và cho B.
3. **Chạy thử toàn bộ quy trình nộp bài** bằng cách giả vờ fold 0 là tập test: train → dự đoán → ghi file → kiểm tra format → chấm.
4. Chuẩn bị sẵn ô lệnh cho tháng 01/2027: khi có file test thật, chỉ cần điền đường dẫn rồi chạy.

**Nguyên tắc trung thực (lồng nhau).** Mọi lựa chọn (chọn model nào, trọng số bao nhiêu, bias bao nhiêu) đều được học trên 4 fold rồi chấm trên fold thứ 5. Nếu chọn trên toàn bộ OOF rồi chấm lại chính OOF đó, điểm sẽ đẹp hơn thực tế, và đến lúc nộp test sẽ bị "rơi điểm".

In [1]:
# Chạy trên Kaggle? Ô này chép bộ dữ liệu + code (dataset bạn đã upload) sang /kaggle/working
# vì /kaggle/input chỉ đọc được, không ghi được. Trên máy nhà ô này tự bỏ qua.
import os
import shutil
from pathlib import Path

if Path("/kaggle/input").exists():
    bundle = next(Path("/kaggle/input").rglob("SemEval2027_Task8")).parent
    work = Path("/kaggle/working/sqe")
    if not work.exists():
        shutil.copytree(bundle, work)
    os.chdir(work / "SemEval2027_Task8" / "notebooks")
    print("Kaggle – thư mục làm việc:", Path.cwd())

Kaggle – thư mục làm việc: /kaggle/working/sqe/SemEval2027_Task8/notebooks


In [2]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sqe").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from sqe import classical
from sqe.data import TASKS, load_en
from sqe.decision import apply_bias, average, greedy_select, nested_bias, nested_greedy, tune_bias
from sqe.metrics import fold_scores, paired_delta
from sqe.results import OOF_DIR, load_oof, load_results, log_result
from sqe.submission import read_submission, write_submission
from sqe.train import RUNS_DIR, fit_full_and_predict

pd.set_option("display.precision", 4)
df = load_en()
FOLDS = sorted(df.fold.unique())

## 1. Các model ứng viên

Mọi model đã có OOF với xác suất: TF-IDF và các model fine-tune ở notebook 03–05. Bỏ qua random/majority, các file `_bias` (vì đó là bản đã chỉnh của model khác), và LLM (chỉ có nhãn, không có xác suất, và chỉ chạy trên fold 0).

In [3]:
def candidates(t):
    names = []
    for f in sorted(OOF_DIR.glob(f"*_{t}.csv")):
        name = f.stem[:-2]
        if name.startswith(("random_", "majority_", "tfidf_lr_")) or name.endswith("_bias"):
            continue
        names.append(name)
    return names


CANDS, OOFS = {}, {}
for t in "AB":
    CANDS[t] = candidates(t)
    OOFS[t] = [load_oof(n, t) for n in CANDS[t]]
    ids = OOFS[t][0].StereoQueerEval_id
    assert all((o.StereoQueerEval_id == ids).all() for o in OOFS[t]), "OOF phải cùng thứ tự dòng"
    sc = pd.DataFrame([{"model": n, "macro_f1": fold_scores(o, TASKS[t][1]).macro_f1.mean()}
                       for n, o in zip(CANDS[t], OOFS[t])]).sort_values("macro_f1", ascending=False)
    print(f"Subtask {t}: {len(CANDS[t])} ứng viên")
    display(sc.reset_index(drop=True))

Subtask A: 10 ứng viên


,model,macro_f1
0,mt_twhate_comment,0.7354
1,twhate_comment,0.7335
2,mtcw_twhate_comment,0.7301
3,roberta_pair_title_desc,0.7288
4,roberta_comment,0.7274
5,roberta_pair_title,0.7266
6,roberta_concat_tdc,0.7113
7,tfidf_comment,0.7002
8,tfidf_comment_title,0.6775
9,deberta_comment,0.3575


Subtask B: 10 ứng viên


,model,macro_f1
0,mt_twhate_comment,0.6006
1,mtcw_twhate_comment,0.5982
2,twhate_comment,0.5870
3,roberta_comment,0.5856
4,roberta_pair_title,0.5808
5,roberta_pair_title_desc,0.5692
6,roberta_concat_tdc,0.5506
7,tfidf_comment,0.5249
8,tfidf_comment_title,0.4833
9,deberta_comment,0.2134


## 2. So sánh các chiến lược (tất cả đều lồng nhau)

| Chiến lược | Cách làm (trên 4 fold) → áp dụng lên fold thứ 5 |
|---|---|
| best single | chọn model đơn lẻ có điểm cao nhất |
| top-3 mean | trung bình xác suất của 3 model cao nhất |
| greedy | chọn dần từng model (được chọn lặp lại) sao cho macro-F1 tăng nhiều nhất (Caruana, 2004) |
| … + bias | tinh chỉnh ngưỡng lồng nhau trên kết quả của chiến lược đó |

*Lưu ý nhỏ:* với "+ bias", phần ensemble của các fold khác cũng đã được chọn có dùng fold đang chấm, nên vẫn còn một chút lạc quan. Coi đây là con số tham khảo.

In [4]:
def macro(o, labels, rows=None):
    o = o if rows is None else o[rows]
    return f1_score(o.y_true, o.y_pred, labels=labels, average="macro", zero_division=0)


def nested_topk(oofs, labels, k):
    out = average(oofs, labels)
    pcols = [f"p_{l}" for l in labels]
    for f in FOLDS:
        tr = (out.fold != f).to_numpy()
        top = np.argsort([-macro(o, labels, tr) for o in oofs])[:k]
        out.loc[~tr, pcols] = average([oofs[i] for i in top], labels).loc[~tr, pcols].to_numpy()
    out["y_pred"] = np.array(labels)[out[pcols].to_numpy().argmax(1)]
    return out


STRATS = {}
rows = []
for t in "AB":
    labels = TASKS[t][1]
    res_t = {"best single": nested_topk(OOFS[t], labels, 1), "top-3 mean": nested_topk(OOFS[t], labels, 3),
             "greedy": nested_greedy(OOFS[t], labels)[0]}
    for k in list(res_t):
        res_t[k + " + bias"] = nested_bias(res_t[k], labels)[0]
    STRATS[t] = res_t
    base = fold_scores(res_t["best single"], labels)
    for k, o in res_t.items():
        sc = fold_scores(o, labels)
        d = paired_delta(sc, base)
        rows.append({"subtask": t, "chiến lược": k, "macro_f1 (lồng nhau)": sc.macro_f1.mean(),
                     "std": sc.macro_f1.std(ddof=0), "Δ so với best single": d["delta_mean"],
                     "thắng (fold)": f"{d['wins']}/5",
                     **{c: sc[c].mean() for c in sc.columns if c.startswith("f1_")}})
strat_table = pd.DataFrame(rows)
strat_table

,subtask,chiến lược,macro_f1 (lồng nhau),std,Δ so với best single,thắng (fold),f1_no,f1_yes,f1_yes_implicit,f1_yes_explicit
0,A,best single,0.7323,1.4648e-02,0.0000,0/5,0.7640,0.7007,NaN,NaN
1,A,top-3 mean,0.7419,1.0221e-02,0.0095,4/5,0.7711,0.7126,NaN,NaN
2,A,greedy,0.3575,2.5410e-04,-0.3749,0/5,0.7150,0.0000,NaN,NaN
3,A,best single + bias,0.7353,1.3609e-02,0.0030,4/5,0.7707,0.6999,NaN,NaN
4,A,top-3 mean + bias,0.7419,1.0221e-02,0.0095,4/5,0.7711,0.7126,NaN,NaN
5,A,greedy + bias,0.3575,2.5410e-04,-0.3749,0/5,0.7150,0.0000,NaN,NaN
6,B,best single,0.6006,2.5934e-02,0.0000,0/5,0.7501,NaN,0.4313,0.6204
7,B,top-3 mean,0.6043,1.6352e-02,0.0037,2/5,0.7417,NaN,0.4540,0.6173
8,B,greedy,0.2134,1.4778e-05,-0.3872,0/5,0.6401,NaN,0.0000,0.0000
9,B,best single + bias,0.5962,1.5989e-02,-0.0044,2/5,0.7339,NaN,0.4412,0.6136


## 3. Chốt công thức cho từng subtask

**Quy tắc:** lấy chiến lược có điểm lồng nhau cao nhất. Nhưng nếu nó hơn "best single" chưa tới 0,005 thì chọn "best single" (hoặc "best single + bias" nếu bản đó hơn), vì hệ thống đơn giản thì dễ tái lập và ít rủi ro hơn.

Sau khi chọn được chiến lược, ta học lại các tham số của nó (model nào, trọng số, bias) trên **toàn bộ** OOF để dùng cho tập test.

In [5]:
RECIPE = {}
for t in "AB":
    labels = TASKS[t][1]
    tab = strat_table[strat_table.subtask == t].set_index("chiến lược")["macro_f1 (lồng nhau)"]
    simple = tab[["best single", "best single + bias"]].idxmax()
    pick = tab.idxmax() if tab.max() - tab[simple] >= 0.005 else simple
    full_scores = [macro(o, labels) for o in OOFS[t]]
    if pick.startswith("best single"):
        weights = np.eye(len(OOFS[t]))[int(np.argmax(full_scores))]
    elif pick.startswith("top-3"):
        weights = np.isin(np.arange(len(OOFS[t])), np.argsort(full_scores)[::-1][:3]).astype(float)
    else:
        weights = greedy_select(OOFS[t], labels).astype(float)
    ens = average(OOFS[t], labels, weights)
    bias = tune_bias(ens[[f"p_{l}" for l in labels]].to_numpy(),
                     ens.y_true.map({l: i for i, l in enumerate(labels)}).to_numpy()) if pick.endswith("bias") \
        else np.zeros(len(labels))
    comps = {CANDS[t][i]: float(w) for i, w in enumerate(weights) if w > 0}
    RECIPE[t] = {"strategy": pick, "components": comps, "bias": bias.tolist(),
                 "nested_macro_f1": float(tab[pick])}
    print(f"Subtask {t}: {pick} | thành phần (trọng số): {comps} | bias: {np.round(bias, 2).tolist()} "
          f"| macro-F1 lồng nhau ≈ {tab[pick]:.4f}")
    final_oof = STRATS[t][pick]
    log_result("08", "final: " + pick, "+".join(comps), t, fold_scores(final_oof, labels),
               note="hệ thống cuối, điểm lồng nhau")

(ROOT / "outputs" / "submissions").mkdir(parents=True, exist_ok=True)
(ROOT / "outputs" / "submissions" / "recipe.json").write_text(json.dumps(RECIPE, ensure_ascii=False, indent=1),
                                                              encoding="utf-8")

Subtask A: top-3 mean | thành phần (trọng số): {'mt_twhate_comment': 1.0, 'mtcw_twhate_comment': 1.0, 'twhate_comment': 1.0} | bias: [0.0, 0.0] | macro-F1 lồng nhau ≈ 0.7419
Subtask B: best single | thành phần (trọng số): {'mt_twhate_comment': 1.0} | bias: [0.0, 0.0, 0.0] | macro-F1 lồng nhau ≈ 0.6006


394

## 4. Quy trình dự đoán cho một tập "test" bất kỳ

Hàm `predict_recipe(df_train, df_test, t)`: train lại từng thành phần của công thức trên `df_train` (toàn bộ, không chia fold), dự đoán `df_test`, lấy trung bình có trọng số, cộng bias, rồi lấy argmax.

Model multi-task (tên bắt đầu bằng `mt`) được train một lần và cho ra cả A lẫn B. Kết quả được nhớ lại trong biến `_cache` để không train hai lần.

In [6]:
_cache = {}


def component_cfg(name, t):
    """Đọc cấu hình đã dùng khi chạy CV của thành phần `name` (từ outputs/runs/)."""
    for f in sorted(RUNS_DIR.glob(f"{name}_*.json")):
        if f.stem[len(name) + 1:] in ("A", "B", "AB"):
            meta = json.loads(f.read_text(encoding="utf-8"))
            if t in meta["tasks"]:
                return meta["cfg"], tuple(meta["tasks"])
    raise FileNotFoundError(name)


def predict_component(name, t, df_train, df_test, tag):
    labels = TASKS[t][1]
    if name.startswith("tfidf_"):
        texts = {"tfidf_comment": lambda d: d.yt_comment,
                 "tfidf_comment_title": lambda d: d.yt_comment + "\n" + d.yt_title}[name]
        m = classical.make_model("tfidf_lr").fit(texts(df_train).tolist(), df_train[TASKS[t][0]])
        p = m.predict_proba(texts(df_test).tolist())
        return p[:, [list(m.classes_).index(l) for l in labels]]
    cfg, tasks = component_cfg(name, t)
    key = (tag, name, tasks)
    if key not in _cache:
        print(f"  train {name} {tasks} trên {len(df_train)} mẫu ...")
        _cache[key] = fit_full_and_predict(df_train, df_test, cfg, tasks, log=lambda *_: None)
    return _cache[key][t]


def predict_recipe(df_train, df_test, t, tag):
    labels = TASKS[t][1]
    r = RECIPE[t]
    P = sum(w * predict_component(n, t, df_train, df_test, tag) for n, w in r["components"].items())
    P = apply_bias(P / sum(r["components"].values()), np.array(r["bias"]))
    return np.array(labels)[P.argmax(1)]

## 5. Chạy thử toàn bộ quy trình: giả vờ fold 0 là tập test

- "Train" = fold 1–4, "test" = fold 0 với **cột nhãn bị xoá** (giống file test thật).
- Ghi file nộp, đọc lại, kiểm tra format, rồi chấm bằng nhãn thật của fold 0.
- Mục đích chính là **bắt lỗi quy trình**, không phải để đo điểm: công thức đã được chọn có nhìn cả fold 0, nên điểm ở đây hơi lạc quan.

In [7]:
dry_train = df[df.fold != 0].reset_index(drop=True)
dry_test = df[df.fold == 0].reset_index(drop=True)
truth = dry_test[["StereoQueerEval_id", "stereotype", "hate_speech"]].copy()
dry_test = dry_test.drop(columns=["stereotype", "hate_speech", "target"])

SUB = ROOT / "outputs" / "submissions"
dry_rows = []
for t in "AB":
    col, labels = TASKS[t]
    pred = predict_recipe(dry_train, dry_test, t, tag="dry")
    path = write_submission(dry_test.StereoQueerEval_id, pred, t, SUB / f"dryrun_fold0_{t}.tsv")
    back = read_submission(path, t)
    assert list(back.StereoQueerEval_id) == list(truth.StereoQueerEval_id)
    s = f1_score(truth[col], back.label, labels=labels, average="macro")
    nested_f0 = fold_scores(STRATS[t][RECIPE[t]["strategy"]], labels).set_index("fold").macro_f1[0]
    dry_rows.append({"subtask": t, "file": path.name, "dòng": len(back), "macro_f1 fold 0 (chạy thử)": s,
                     "macro_f1 fold 0 (OOF lồng nhau)": nested_f0})
display(pd.DataFrame(dry_rows))
print("5 dòng đầu file nộp B:")
print("".join(open(SUB / "dryrun_fold0_B.tsv", encoding="utf-8").readlines()[:5]))

  train mt_twhate_comment ('A', 'B') trên 2390 mẫu ...
  train mtcw_twhate_comment ('A', 'B') trên 2390 mẫu ...
  train twhate_comment ('A',) trên 2390 mẫu ...


,subtask,file,dòng,macro_f1 fold 0 (chạy thử),macro_f1 fold 0 (OOF lồng nhau)
0,A,dryrun_fold0_A.tsv,599,0.7414,0.7417
1,B,dryrun_fold0_B.tsv,599,0.6111,0.6111


5 dòng đầu file nộp B:
training_EN_0003	yes_explicit
training_EN_0005	yes_implicit
training_EN_0010	yes_explicit
training_EN_0013	no
training_EN_0020	no



Hai cột điểm không cần khớp tuyệt đối: OOF lồng nhau dùng model train trên các fold khác, có lấy epoch cuối và cùng seed. Nhưng nếu lệch nhau quá xa (ví dụ > 0,05) thì quy trình có lỗi cần xem lại.

## 6. Khi có tập test thật (tháng 01/2027)

1. Tải file test của ban tổ chức, giải nén (mật khẩu theo email của BTC), đặt vào `StereoQueerEval_SemEval-2027/`.
2. Điền đường dẫn vào `TEST_PATH` bên dưới rồi chạy ô này. Mất khoảng 10–30 phút trên GPU.
3. Nộp `outputs/submissions/test_EN_A.tsv` lên trang Codabench của Subtask A, và `test_EN_B.tsv` lên trang của Subtask B.
4. Trước khi nộp: kiểm tra trên Codabench xem có yêu cầu header hoặc tên file cụ thể không. Nếu có, sửa `HEADER` trong `src/sqe/submission.py`.

In [8]:
TEST_PATH = ROOT.parent / "StereoQueerEval_SemEval-2027" / "StereoQueerEval_EN_test.tsv"   # ← sửa khi có file thật

if TEST_PATH.exists():
    test = pd.read_csv(TEST_PATH, sep="\t", dtype=str, keep_default_na=False, encoding="utf-8")
    assert {"StereoQueerEval_id", "yt_title", "yt_description", "yt_comment"} <= set(test.columns)
    print("Số mẫu test:", len(test), "(guideline: 743)")
    for t in "AB":
        pred = predict_recipe(df, test, t, tag="test")
        p = write_submission(test.StereoQueerEval_id, pred, t, SUB / f"test_EN_{t}.tsv")
        print("Đã ghi", p, "| phân bố nhãn:", pd.Series(pred).value_counts(normalize=True).round(3).to_dict())
else:
    print(f"Chưa có file test ({TEST_PATH.name}). Bỏ qua — ô này dành cho tháng 01/2027.")

Chưa có file test (StereoQueerEval_EN_test.tsv). Bỏ qua — ô này dành cho tháng 01/2027.


## 7. Bảng tổng kết cho báo cáo

Mỗi subtask lấy một dòng đại diện cho từng bước của roadmap. Tất cả là macro-F1 trung bình 5 fold theo video, trừ dòng LLM (chỉ fold 0).

In [9]:
res = load_results()
pick = []
for t in "AB":
    r = res[res.subtask == t]

    def best(mask, label):
        s = r[mask]
        if len(s):
            row = s.sort_values("macro_f1").iloc[-1]
            pick.append({"subtask": t, "bước": label, "model": row.model, "input": row.input,
                         "macro_f1": row.macro_f1, "std": row.macro_f1_std, "ghi chú": row.note})

    best(r.model == "majority", "majority (mốc sàn)")
    best(r.model == "tfidf_lr", "TF-IDF + LR (notebook 02/03)")
    best((r.notebook == "03") & (r.model == "roberta-base"), "roberta-base, comment (03)")
    best((r.notebook == "04") & (r.model == "roberta-base"), "roberta-base + context tốt nhất (04)")
    best((r.notebook == "04") & (r.model != "roberta-base"), "đổi encoder tốt nhất (04)")
    best(r.notebook == "05", "multi-task / bias tốt nhất (05)")
    best(r.notebook == "06", "Qwen2.5-7B (06, chỉ fold 0)")
    best(r.notebook == "08", "hệ thống cuối (08, lồng nhau)")
summary = pd.DataFrame(pick)
summary.to_csv(ROOT / "outputs" / "results" / "summary_table.csv", index=False, encoding="utf-8-sig")
summary

,subtask,bước,model,input,macro_f1,std,ghi chú
0,A,majority (mốc sàn),majority,comment,0.3575,2.5410e-04,baseline cổ điển (cấu hình notebook 02)
1,A,TF-IDF + LR (notebook 02/03),tfidf_lr,comment,0.7002,2.0919e-02,baseline cổ điển (cấu hình notebook 02)
2,A,"roberta-base, comment (03)",roberta-base,comment,0.7274,9.7725e-03,lr2e-5 ep4 bs16 len128
3,A,roberta-base + context tốt nhất (04),roberta-base,pair_title_desc,0.7288,5.2874e-03,"C2, len256"
4,A,đổi encoder tốt nhất (04),cardiffnlp/twitter-roberta-base-hate-latest,comment,0.7335,1.1522e-02,"đổi encoder, len128"
5,A,multi-task / bias tốt nhất (05),cardiffnlp/twitter-roberta-base-hate-latest,comment,0.7352,1.0885e-02,MT + cw + bias lồng nhau
6,A,"hệ thống cuối (08, lồng nhau)",final: top-3 mean,mt_twhate_comment+mtcw_twhate_comment+twhate_c...,0.7419,1.0221e-02,"hệ thống cuối, điểm lồng nhau"
7,B,majority (mốc sàn),majority,comment,0.2134,1.4778e-05,baseline cổ điển (cấu hình notebook 02)
8,B,TF-IDF + LR (notebook 02/03),tfidf_lr,comment,0.5249,1.3831e-02,baseline cổ điển (cấu hình notebook 02)
9,B,"roberta-base, comment (03)",roberta-base,comment,0.5856,1.7904e-02,lr2e-5 ep4 bs16 len128


## 8. Kết luận

*(phần này được viết sau khi chạy xong, dựa trên số liệu ở trên)*